# **Parallelization in LangGraph**

LangGraph can run multiple nodes **simultaneously** when they all connect from the same upstream node.
Here, three post-creation nodes (Instagram, Twitter, LinkedIn) all branch from START and run in parallel.

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

llm.invoke("I want to know the meaning of water").content

### **Schema**
All three post fields live in one state dict; each node fills in its own field.

In [ ]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    topic:    str   # shared input — the topic for all three posts
    insta:    str   # filled by create_post_insta
    twitter:  str   # filled by create_post_twitter
    linkedin: str   # filled by create_post_linkedin

### **Three Independent Node Functions**
Each reads only `topic` from state and writes back only its own field using a partial return dict.

In [ ]:
def create_post_insta(state: graph_schema) -> graph_schema:
    topic = state['topic']
    post  = llm.invoke(f"Write an Instagram post about {topic}. Keep the tone casual and engaging.").content
    # Returning a partial dict is fine — LangGraph merges it into the full state
    return {'insta': post}


def create_post_twitter(state: graph_schema) -> graph_schema:
    topic = state['topic']
    post  = llm.invoke(f"Write a Twitter post about {topic}. Keep the tone quick.").content
    return {'twitter': post}


def create_post_linkedin(state: graph_schema) -> graph_schema:
    topic = state['topic']
    post  = llm.invoke(f"Write a LinkedIn post about {topic}. Keep the tone professional and informative.").content
    return {'linkedin': post}

### **Build the Parallel Graph**
Adding edges from START to multiple nodes tells LangGraph to run all of them concurrently.

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image

graph = StateGraph(graph_schema)

graph.add_node("create_post_insta",    create_post_insta)
graph.add_node("create_post_twitter",  create_post_twitter)
graph.add_node("create_post_linkedin", create_post_linkedin)

# START fans out to all three nodes simultaneously
graph.add_edge(START, "create_post_insta")
graph.add_edge(START, "create_post_twitter")
graph.add_edge(START, "create_post_linkedin")

# Each node independently flows to END
graph.add_edge("create_post_insta",    END)
graph.add_edge("create_post_twitter",  END)
graph.add_edge("create_post_linkedin", END)

parallel_graph = graph.compile()
Image(parallel_graph.get_graph().draw_mermaid_png())

In [ ]:
result = parallel_graph.invoke({
    "topic":    "Artificial Intelligence",
    "insta":    "",
    "twitter":  "",
    "linkedin": ""
})

print("Instagram:", result['insta'][:80], "...")
print()
print("Twitter:",   result['twitter'][:80], "...")
print()
print("LinkedIn:",  result['linkedin'][:80], "...")